### import libraries

In [1]:
import sys
sys.path.append("..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src import config, db
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
sns.set_style("whitegrid")
%matplotlib inline

### load data

In [2]:
client = db.get_client()
collection = db.get_collection(config.FEATURES_COLLECTION, client)
cursor = collection.find({"city": config.CITY_NAME}).sort("timestamp", 1)
df = pd.DataFrame(list(cursor))
client.close()
df = df.drop(columns=["nh3", "_id"])
df = df.sort_values("timestamp").reset_index(drop=True)
print(df.shape)

(17544, 21)


### leak-safe feature builder

In [3]:
TRAIN_SPLIT_RATIO = 0.85

def build_target_and_split(df, horizon_hours):
    n = len(df)
    current = df.iloc[: n - horizon_hours].reset_index(drop=True)
    future = df.iloc[horizon_hours:].reset_index(drop=True)

    merged = pd.DataFrame({
        "current_timestamp": current["timestamp"].values,
        "future_timestamp": future["timestamp"].values,
        "aqi": current["aqi"].values, "pm2_5": current["pm2_5"].values,
        "pm10": current["pm10"].values, "co": current["co"].values,
        "no2": current["no2"].values, "so2": current["so2"].values,
        "o3": current["o3"].values, "aqi_change_rate": current["aqi_change_rate"].values,
        "temperature": future["temperature"].values, "humidity": future["humidity"].values,
        "pressure": future["pressure"].values, "wind_speed": future["wind_speed"].values,
        "hour": future["hour"].values, "day": future["day"].values,
        "month": future["month"].values, "day_of_week": future["day_of_week"].values,
        "aqi_target": future["aqi"].values,
    })

    expected_future = pd.to_datetime(merged["current_timestamp"]) + pd.Timedelta(hours=horizon_hours)
    aligned_mask = pd.to_datetime(merged["future_timestamp"]) == expected_future
    merged = merged[aligned_mask].reset_index(drop=True)

    split_index = int(len(merged) * TRAIN_SPLIT_RATIO)
    return merged.iloc[:split_index], merged.iloc[split_index:]

FEATURE_COLS = ["aqi", "pm2_5", "pm10", "co", "no2", "so2", "o3", "aqi_change_rate",
                "temperature", "humidity", "pressure", "wind_speed",
                "hour", "day", "month", "day_of_week"]

### compare 5 algorithms on Day 1 (24h)

In [7]:
train_df, test_df = build_target_and_split(df, 24)
X_train, y_train = train_df[FEATURE_COLS], train_df["aqi_target"]
X_test, y_test = test_df[FEATURE_COLS], test_df["aqi_target"]

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f"{name}: RMSE={rmse:.2f}, MAE={mae:.2f}, R2={r2:.3f}")
    results.append({"model": name, "RMSE": rmse, "MAE": mae, "R2": r2})

pd.DataFrame(results)

Linear Regression: RMSE=13.04, MAE=9.13, R2=0.718
Ridge Regression: RMSE=13.04, MAE=9.13, R2=0.718
Random Forest: RMSE=15.08, MAE=9.69, R2=0.624
Gradient Boosting: RMSE=14.86, MAE=9.50, R2=0.634
XGBoost: RMSE=14.81, MAE=9.56, R2=0.637


,model,RMSE,MAE,R2
0,Linear Regression,13.044198,9.128163,0.718487
1,Ridge Regression,13.044201,9.128169,0.718487
2,Random Forest,15.077463,9.692233,0.623885
3,Gradient Boosting,14.864756,9.497741,0.634422
4,XGBoost,14.810771,9.564838,0.637073


### 48 hours

In [8]:
train_df, test_df = build_target_and_split(df, 48)
X_train, y_train = train_df[FEATURE_COLS], train_df["aqi_target"]
X_test, y_test = test_df[FEATURE_COLS], test_df["aqi_target"]

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f"{name}: RMSE={rmse:.2f}, MAE={mae:.2f}, R2={r2:.3f}")
    results.append({"model": name, "RMSE": rmse, "MAE": mae, "R2": r2})

pd.DataFrame(results)

Linear Regression: RMSE=19.71, MAE=13.53, R2=0.358
Ridge Regression: RMSE=19.71, MAE=13.53, R2=0.358
Random Forest: RMSE=18.92, MAE=12.27, R2=0.409
Gradient Boosting: RMSE=19.36, MAE=12.74, R2=0.380
XGBoost: RMSE=19.50, MAE=12.78, R2=0.372


,model,RMSE,MAE,R2
0,Linear Regression,19.712377,13.527359,0.357615
1,Ridge Regression,19.712388,13.527374,0.357615
2,Random Forest,18.915462,12.266654,0.408505
3,Gradient Boosting,19.361369,12.740437,0.380289
4,XGBoost,19.496187,12.779908,0.371629


### 72 hours

In [9]:
train_df, test_df = build_target_and_split(df, 72)
X_train, y_train = train_df[FEATURE_COLS], train_df["aqi_target"]
X_test, y_test = test_df[FEATURE_COLS], test_df["aqi_target"]

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f"{name}: RMSE={rmse:.2f}, MAE={mae:.2f}, R2={r2:.3f}")
    results.append({"model": name, "RMSE": rmse, "MAE": mae, "R2": r2})

pd.DataFrame(results)

Linear Regression: RMSE=23.14, MAE=15.93, R2=0.116
Ridge Regression: RMSE=23.14, MAE=15.93, R2=0.116
Random Forest: RMSE=22.76, MAE=14.99, R2=0.144
Gradient Boosting: RMSE=21.55, MAE=14.61, R2=0.233
XGBoost: RMSE=21.45, MAE=14.51, R2=0.240


,model,RMSE,MAE,R2
0,Linear Regression,23.138155,15.927907,0.116118
1,Ridge Regression,23.138171,15.927925,0.116116
2,Random Forest,22.764865,14.988856,0.144407
3,Gradient Boosting,21.548709,14.606420,0.233381
4,XGBoost,21.448601,14.512990,0.240487
